I want to compare all of the models we train here.

We will fit on a unified split using each model's best hyperparameters, this gives us the best fair comparison .

In [41]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score
import plotly.graph_objects as go

In [42]:
df = pd.read_csv("../data/drug_discovery_virtual_screening_cleaned.csv")
df = df.drop(columns=["Unnamed: 0"])


X = df.drop(columns=["active"])
y = df["active"]

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, stratify=y, random_state=42
)

In [43]:
scaler = StandardScaler().set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 1599
Testing samples: 400


In [44]:
# we need to specify for each whether or not it should be fit on scaled or unscaled features. 
models = {
    "Dummy (baseline)":    (DummyClassifier(strategy="most_frequent"), "scaled"),
    "KNN (k=41, cosine)":  (KNeighborsClassifier(n_neighbors=41, metric="cosine"), "scaled"),
    "Decision Tree (d=4)": (DecisionTreeClassifier(max_depth=4, random_state=42), "unscaled"),
    "Random Forest":       (RandomForestClassifier(n_estimators=100, max_depth=None, min_samples_split=2, random_state=42), "unscaled"),
    "Gradient Boosting":   (GradientBoostingClassifier(learning_rate=1.0, max_depth=3, n_estimators=3, random_state=42), "unscaled"),
    "LogReg L2 (C=10)":    (LogisticRegression(penalty="l2", solver="liblinear", C=10, max_iter=10000), "scaled"),
    "LogReg L1 (C=0.01)":  (LogisticRegression(penalty="l1", solver="liblinear", C=0.01, max_iter=10000), "scaled"),
}

In [45]:
# Here we fit each model 
results = []
fitted = {}

for name, (model, scaling) in models.items():
    if scaling == "scaled":
        Xtrain, Xtest = X_train_scaled, X_test_scaled
    else:
        Xtrain, Xtest = X_train, X_test

    model.fit(Xtrain, y_train)
    y_pred = model.predict(Xtest)
    y_prob = model.predict_proba(Xtest)[:, 1]

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_prob)

    results.append({"model": name, "accuracy": acc, "roc_auc": auc})
    fitted[name] = {"fpr": fpr, "tpr": tpr, "auc": auc, "y_prob": y_prob}
    
results_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
results_df

,model,accuracy,roc_auc
0,LogReg L2 (C=10),0.8900,0.958043
1,LogReg L1 (C=0.01),0.8900,0.954476
2,Random Forest,0.8850,0.948947
3,Decision Tree (d=4),0.8950,0.938303
4,"KNN (k=41, cosine)",0.8825,0.936372
5,Gradient Boosting,0.8800,0.932185
6,Dummy (baseline),0.6950,0.500000


## ROC curve overlay

In [46]:
fig = go.Figure()
for name in results_df["model"]:
    info = fitted[name]
    label = f"{name} (AUC = {info['auc']:.3f})"
    fig.add_trace(go.Scatter(x=info["fpr"], y=info["tpr"], mode="lines", name=label))

fig.update_layout(
    title="ROC Curves For All Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate (Recall)",
    width=800,
    height=700
)
fig.show()

## Threshold tuning across models

In [47]:
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, average_precision_score

thresholds = np.linspace(0.05, 0.95, 91)
threshold_results = {}

for name in fitted:
    y_prob_model = fitted[name].get("y_prob")
    if not y_prob_model is None:
        rows = []
        for t in thresholds:
            preds = (y_prob_model >= t).astype(int)
            rows.append({
                "threshold": t,
                "precision": precision_score(y_test, preds),
                "recall": recall_score(y_test, preds),
                "f1": f1_score(y_test, preds),
            })
        threshold_results[name] = pd.DataFrame(rows)

In [48]:
summary_rows = []
for name, thresholds in threshold_results.items():
    if name == "Dummy (baseline)":
        continue
    
    y_prob_model = fitted[name]["y_prob"]
    
    default_row = thresholds.iloc[(thresholds["threshold"] - 0.5).abs().idxmin()]
    
    best_f1 = thresholds.loc[thresholds["f1"].idxmax()]
    
    high_recall = thresholds[thresholds["recall"] >= 0.9]
    if len(high_recall) > 0:
        best_recall = high_recall.sort_values("threshold").iloc[-1]
    else:
        best_recall = None
    
    summary_rows.append({
        "model": name,
        "AP": round(average_precision_score(y_test, y_prob_model), 3),

        "default precision": round(default_row["precision"], 3),
        "default recall": round(default_row["recall"], 3),
        "default F1": round(default_row["f1"], 3),

        "F1-opt threshold": round(best_f1["threshold"], 2),
        "F1-opt precision": round(best_f1["precision"], 3),
        "F1-opt recall": round(best_f1["recall"], 3),
        "F1-opt F1": round(best_f1["f1"], 3),

        "recall=0.9 threshold": round(best_recall["threshold"], 2) if best_recall is not None else None,
        "recall=0.9 precision": round(best_recall["precision"], 3) if best_recall is not None else None,
    })

pd.DataFrame(summary_rows).sort_values("F1-opt F1", ascending=False).reset_index(drop=True)

,model,AP,default precision,default recall,default F1,F1-opt threshold,F1-opt precision,F1-opt recall,F1-opt F1,recall=0.9 threshold,recall=0.9 precision
0,LogReg L2 (C=10),0.928,0.861,0.762,0.809,0.30,0.829,0.877,0.853,0.25,0.764
1,LogReg L1 (C=0.01),0.922,0.861,0.762,0.809,0.42,0.817,0.877,0.846,0.38,0.755
2,Random Forest,0.908,0.858,0.746,0.798,0.29,0.809,0.869,0.838,0.20,0.728
3,Decision Tree (d=4),0.869,0.877,0.762,0.816,0.21,0.821,0.828,0.824,0.20,0.745
4,"KNN (k=41, cosine)",0.898,0.895,0.697,0.783,0.40,0.843,0.795,0.819,0.29,0.683
5,Gradient Boosting,0.889,0.863,0.721,0.786,0.12,0.780,0.844,0.811,0.10,0.719


## Precision-Recall curve overlay

In [49]:
fig = go.Figure()
for name in results_df["model"]:
    if name == "Dummy (baseline)":
        continue
    y_prob_model = fitted[name]["y_prob"]
    precisions, recalls, _ = precision_recall_curve(y_test, y_prob_model)
    ap = average_precision_score(y_test, y_prob_model)
    label = f"{name} (AP = {ap:.3f})"
    fig.add_trace(go.Scatter(x=recalls, y=precisions, mode="lines", name=label))

fig.update_layout(
    title="PR Curves For All Models",
    xaxis_title="Recall",
    yaxis_title="Precision",
    width=1000,
    height=700
)
fig.show()